# 🎮 VR World Recommendation System: Enhanced Analytics Version
## Complete Analysis with All Visualizations and Metrics

This enhanced notebook includes:
1. ✅ User & World Concentration Analysis
2. ✅ Action Type & Session Duration Distributions
3. ✅ Engagement Patterns & Category Performance
4. ✅ Interaction Matrix Visualization
5. ✅ User Embeddings with t-SNE
6. ✅ Comprehensive Feature Analysis
7. ✅ Complete Model Comparison with All Metrics
8. ✅ Performance Radar Charts

In [ ]:
# === Comprehensive Imports ===
import os, random
import numpy as np, pandas as pd
from scipy import sparse
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_score, 
    recall_score, ndcg_score, dcg_score
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Polygon
import warnings
warnings.filterwarnings('ignore')

# Optional imports
try:
    import seaborn as sns
    sns.set(style="whitegrid", font_scale=1.2)
except ImportError:
    print("Warning: seaborn not installed. Using matplotlib only.")
    sns = None

plt.rcParams["figure.figsize"] = (12, 8)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

print("✅ Enhanced environment ready!")

## Step 1: Data Loading and Initial Exploration

In [ ]:
# === Load Data ===
for fn in ['users.csv', 'vrworld.csv', 'user_actions.csv', 'time_spent.csv']:
    if not os.path.exists(fn):
        raise FileNotFoundError(f"Missing file: {fn}")

users = pd.read_csv('users.csv', dtype=str)
worlds = pd.read_csv('vrworld.csv', dtype=str)
actions = pd.read_csv('user_actions.csv', dtype=str)
time_spent = pd.read_csv('time_spent.csv', dtype=str)

# Normalize column names
for df in [users, worlds, actions, time_spent]:
    df.columns = df.columns.str.strip().str.lower()

print("📊 Dataset Shapes:")
print(f"  Users: {users.shape}")
print(f"  Worlds: {worlds.shape}")
print(f"  Actions: {actions.shape}")
print(f"  Time Spent: {time_spent.shape}")

# Calculate sparsity
n_unique_users = users['user_id'].nunique()
n_unique_worlds = worlds['world_id'].nunique()
n_interactions = actions[['user_id', 'world_id']].drop_duplicates().shape[0]
sparsity = 1 - (n_interactions / (n_unique_users * n_unique_worlds))

print(f"\n📈 Key Statistics:")
print(f"  Unique users: {n_unique_users:,}")
print(f"  Unique worlds: {n_unique_worlds:,}")
print(f"  Unique interactions: {n_interactions:,}")
print(f"  Sparsity: {sparsity:.1%}")

## Step 2: User & World Concentration Analysis

In [ ]:
# === User and World Concentration Charts ===
user_activity = actions.groupby('user_id').size().reset_index(name='num_actions')
world_popularity = actions.groupby('world_id').size().reset_index(name='num_visitors')

# Calculate concentration metrics
user_sorted = user_activity.sort_values('num_actions', ascending=False)
world_sorted = world_popularity.sort_values('num_visitors', ascending=False)

# Cumulative percentages
user_sorted['cumsum_pct'] = user_sorted['num_actions'].cumsum() / user_sorted['num_actions'].sum()
world_sorted['cumsum_pct'] = world_sorted['num_visitors'].cumsum() / world_sorted['num_visitors'].sum()

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# User Concentration (Lorenz Curve)
ax1 = axes[0, 0]
user_pct = np.arange(1, len(user_sorted) + 1) / len(user_sorted)
ax1.plot(user_pct, user_sorted['cumsum_pct'], 'b-', linewidth=2, label='Actual')
ax1.plot([0, 1], [0, 1], 'r--', linewidth=1, label='Perfect Equality')
ax1.fill_between(user_pct, user_sorted['cumsum_pct'], user_pct, alpha=0.3)
ax1.set_xlabel('Cumulative % of Users')
ax1.set_ylabel('Cumulative % of Actions')
ax1.set_title('User Concentration (Lorenz Curve)')
ax1.grid(True, alpha=0.3)
ax1.legend()

# Calculate Gini coefficient for users
gini_user = (user_sorted['cumsum_pct'].sum() / len(user_sorted) - 0.5) * 2
ax1.text(0.6, 0.2, f'Gini: {gini_user:.3f}', fontsize=12, bbox=dict(boxstyle='round', facecolor='wheat'))

# World Concentration (Lorenz Curve)
ax2 = axes[0, 1]
world_pct = np.arange(1, len(world_sorted) + 1) / len(world_sorted)
ax2.plot(world_pct, world_sorted['cumsum_pct'], 'g-', linewidth=2, label='Actual')
ax2.plot([0, 1], [0, 1], 'r--', linewidth=1, label='Perfect Equality')
ax2.fill_between(world_pct, world_sorted['cumsum_pct'], world_pct, alpha=0.3, color='green')
ax2.set_xlabel('Cumulative % of Worlds')
ax2.set_ylabel('Cumulative % of Visits')
ax2.set_title('World Concentration (Lorenz Curve)')
ax2.grid(True, alpha=0.3)
ax2.legend()

# Calculate Gini coefficient for worlds
gini_world = (world_sorted['cumsum_pct'].sum() / len(world_sorted) - 0.5) * 2
ax2.text(0.6, 0.2, f'Gini: {gini_world:.3f}', fontsize=12, bbox=dict(boxstyle='round', facecolor='lightgreen'))

# Top Users Distribution
ax3 = axes[1, 0]
top_n = 20
top_users = user_sorted.head(top_n)
ax3.bar(range(top_n), top_users['num_actions'], color='steelblue', alpha=0.7)
ax3.set_xlabel('User Rank')
ax3.set_ylabel('Number of Actions')
ax3.set_title(f'Top {top_n} Most Active Users')
ax3.grid(True, alpha=0.3, axis='y')

# Top Worlds Distribution
ax4 = axes[1, 1]
top_worlds = world_sorted.head(top_n)
ax4.bar(range(top_n), top_worlds['num_visitors'], color='darkorange', alpha=0.7)
ax4.set_xlabel('World Rank')
ax4.set_ylabel('Number of Visitors')
ax4.set_title(f'Top {top_n} Most Popular Worlds')
ax4.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("📊 Concentration Metrics:")
print(f"  User Gini Coefficient: {gini_user:.3f}")
print(f"  World Gini Coefficient: {gini_world:.3f}")
print(f"  Top 10% users: {user_sorted.head(int(len(user_sorted)*0.1))['num_actions'].sum() / user_sorted['num_actions'].sum():.1%} of actions")
print(f"  Top 10% worlds: {world_sorted.head(int(len(world_sorted)*0.1))['num_visitors'].sum() / world_sorted['num_visitors'].sum():.1%} of visits")

## Step 3: Action Type & Session Duration Distributions

In [ ]:
# === Action Type Distribution & Session Analysis ===
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Action Type Distribution
ax1 = axes[0, 0]
action_counts = actions['action_type'].value_counts()
colors = plt.cm.Set3(np.linspace(0, 1, len(action_counts)))
wedges, texts, autotexts = ax1.pie(action_counts.values, labels=action_counts.index, 
                                    autopct='%1.1f%%', colors=colors, startangle=90)
ax1.set_title('Action Type Distribution')

# Session Duration Distribution
ax2 = axes[0, 1]
time_spent['avg_time_minutes'] = pd.to_numeric(time_spent['avg_time_minutes'], errors='coerce')
session_durations = time_spent['avg_time_minutes'].dropna()
ax2.hist(session_durations, bins=50, edgecolor='black', alpha=0.7, color='teal')
ax2.set_xlabel('Session Duration (minutes)')
ax2.set_ylabel('Frequency')
ax2.set_title('Session Duration Distribution')
ax2.axvline(session_durations.median(), color='red', linestyle='--', 
           label=f'Median: {session_durations.median():.1f} min')
ax2.legend()

# Action Value Distribution
ax3 = axes[0, 2]
actions['action_value'] = pd.to_numeric(actions['action_value'], errors='coerce')
action_values = actions['action_value'].dropna()
ax3.hist(action_values, bins=30, edgecolor='black', alpha=0.7, color='purple')
ax3.set_xlabel('Action Value')
ax3.set_ylabel('Frequency')
ax3.set_title('Action Value Distribution')

# Actions per User Distribution (Log scale)
ax4 = axes[1, 0]
ax4.hist(user_activity['num_actions'], bins=50, edgecolor='black', alpha=0.7, color='navy')
ax4.set_xlabel('Number of Actions')
ax4.set_ylabel('Number of Users')
ax4.set_title('Actions per User Distribution')
ax4.set_yscale('log')

# Time Spent by Action Type
ax5 = axes[1, 1]
action_time = actions.merge(time_spent, on=['user_id', 'world_id'], how='left')
action_time_grouped = action_time.groupby('action_type')['avg_time_minutes'].mean().sort_values()
ax5.barh(range(len(action_time_grouped)), action_time_grouped.values, color='coral')
ax5.set_yticks(range(len(action_time_grouped)))
ax5.set_yticklabels(action_time_grouped.index)
ax5.set_xlabel('Average Time (minutes)')
ax5.set_title('Average Session Time by Action Type')

# Daily Activity Pattern (simulated)
ax6 = axes[1, 2]
# Simulate hourly pattern
hours = np.arange(24)
activity = np.random.poisson(100, 24) * (1 + 0.5*np.sin(2*np.pi*(hours-14)/24))
ax6.plot(hours, activity, 'o-', color='darkgreen', linewidth=2, markersize=6)
ax6.fill_between(hours, activity, alpha=0.3, color='lightgreen')
ax6.set_xlabel('Hour of Day')
ax6.set_ylabel('Activity Level')
ax6.set_title('Daily Activity Pattern (Simulated)')
ax6.set_xticks([0, 6, 12, 18, 24])
ax6.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("📊 Action & Session Statistics:")
print(f"  Most common action: {action_counts.index[0]} ({action_counts.values[0]:,} occurrences)")
print(f"  Average session duration: {session_durations.mean():.1f} minutes")
print(f"  Median session duration: {session_durations.median():.1f} minutes")
print(f"  95th percentile duration: {session_durations.quantile(0.95):.1f} minutes")

## Step 4: Engagement Patterns & World Categories Performance

In [ ]:
# === Engagement Patterns & Category Analysis ===
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# User Engagement Levels
ax1 = axes[0, 0]
engagement_levels = pd.cut(user_activity['num_actions'], 
                          bins=[0, 5, 20, 50, 100, float('inf')],
                          labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])
engagement_counts = engagement_levels.value_counts()
colors = ['#ff4444', '#ff8844', '#ffcc44', '#88dd44', '#44dd44']
ax1.bar(range(len(engagement_counts)), engagement_counts.values, 
       tick_label=engagement_counts.index, color=colors)
ax1.set_xlabel('Engagement Level')
ax1.set_ylabel('Number of Users')
ax1.set_title('User Engagement Distribution')
ax1.grid(True, alpha=0.3, axis='y')

# World Categories Performance
ax2 = axes[0, 1]
category_stats = worlds.groupby('category').agg({
    'world_id': 'count',
    'rating': lambda x: pd.to_numeric(x, errors='coerce').mean(),
    'likes': lambda x: pd.to_numeric(x, errors='coerce').sum()
}).rename(columns={'world_id': 'count', 'rating': 'avg_rating', 'likes': 'total_likes'})

# Normalize for visualization
category_stats_norm = category_stats.copy()
for col in category_stats_norm.columns:
    category_stats_norm[col] = (category_stats_norm[col] - category_stats_norm[col].min()) / \
                               (category_stats_norm[col].max() - category_stats_norm[col].min())

x = np.arange(len(category_stats_norm))
width = 0.25
ax2.bar(x - width, category_stats_norm['count'], width, label='World Count', color='skyblue')
ax2.bar(x, category_stats_norm['avg_rating'], width, label='Avg Rating', color='gold')
ax2.bar(x + width, category_stats_norm['total_likes'], width, label='Total Likes', color='lightcoral')
ax2.set_xlabel('Category')
ax2.set_ylabel('Normalized Value')
ax2.set_title('World Categories Performance')
ax2.set_xticks(x)
ax2.set_xticklabels(category_stats_norm.index, rotation=45, ha='right')
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

# User-World Interaction Heatmap Sample
ax3 = axes[0, 2]
# Sample 20 users and 20 worlds for visualization
sample_users = np.random.choice(users['user_id'].unique(), 20, replace=False)
sample_worlds = np.random.choice(worlds['world_id'].unique(), 20, replace=False)
sample_actions = actions[
    actions['user_id'].isin(sample_users) & 
    actions['world_id'].isin(sample_worlds)
]
interaction_pivot = sample_actions.pivot_table(
    index='user_id', columns='world_id', values='action_value', 
    aggfunc='count', fill_value=0
)
im = ax3.imshow(interaction_pivot.values, cmap='YlOrRd', aspect='auto')
ax3.set_xlabel('World ID (sample)')
ax3.set_ylabel('User ID (sample)')
ax3.set_title('User-World Interaction Heatmap (20x20 sample)')
plt.colorbar(im, ax=ax3, label='Interactions')

# Retention Curve (simulated)
ax4 = axes[1, 0]
days = np.arange(0, 31)
retention = 100 * np.exp(-days/10) + 20  # Simulated retention curve
ax4.plot(days, retention, 'o-', color='darkblue', linewidth=2)
ax4.fill_between(days, retention, alpha=0.3)
ax4.set_xlabel('Days Since First Visit')
ax4.set_ylabel('Retention Rate (%)')
ax4.set_title('User Retention Curve (Simulated)')
ax4.grid(True, alpha=0.3)
ax4.set_ylim(0, 105)

# World Popularity Over Time (simulated)
ax5 = axes[1, 1]
time_points = 50
top_5_worlds = world_popularity.head(5)['world_id'].values
for i, world in enumerate(top_5_worlds):
    popularity = np.cumsum(np.random.randn(time_points) + 0.5) + i*10
    ax5.plot(popularity, label=f'World {i+1}', linewidth=2)
ax5.set_xlabel('Time Period')
ax5.set_ylabel('Cumulative Popularity')
ax5.set_title('Top 5 Worlds Popularity Trends (Simulated)')
ax5.legend()
ax5.grid(True, alpha=0.3)

# User Exploration Pattern
ax6 = axes[1, 2]
user_world_counts = actions.groupby('user_id')['world_id'].nunique().sort_values()
exploration_bins = [1, 2, 5, 10, 20, 50, 100, float('inf')]
exploration_labels = ['1', '2-4', '5-9', '10-19', '20-49', '50-99', '100+']
exploration_dist = pd.cut(user_world_counts, bins=exploration_bins, labels=exploration_labels)
exploration_counts = exploration_dist.value_counts().sort_index()
ax6.bar(range(len(exploration_counts)), exploration_counts.values, color='teal', alpha=0.7)
ax6.set_xticks(range(len(exploration_counts)))
ax6.set_xticklabels(exploration_counts.index, rotation=45)
ax6.set_xlabel('Number of Worlds Visited')
ax6.set_ylabel('Number of Users')
ax6.set_title('User Exploration Pattern')
ax6.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("📊 Engagement & Category Insights:")
print(f"  Most engaged users: {(engagement_levels == 'Very High').sum()} users")
print(f"  Average worlds per user: {user_world_counts.mean():.1f}")
print(f"  Median worlds per user: {user_world_counts.median():.0f}")
print(f"  Top category by count: {category_stats['count'].idxmax()}")

## Step 5: Create Interaction Matrix & Prepare Features

In [ ]:
# === Build Interaction Matrix ===
# Create encoders for consistent indexing
user_encoder = LabelEncoder().fit(users['user_id'])
world_encoder = LabelEncoder().fit(worlds['world_id'])

# Encode indices
actions['user_idx'] = user_encoder.transform(actions['user_id'])
actions['world_idx'] = world_encoder.transform(actions['world_id'])

# Create weighted interactions
interactions = actions.groupby(['user_idx', 'world_idx']).size().reset_index(name='action_count')

# Add time spent information
time_spent['avg_time_minutes'] = time_spent['avg_time_minutes'].astype(float)
time_agg = time_spent.groupby(['user_id', 'world_id'])['avg_time_minutes'].mean().reset_index()
time_agg['user_idx'] = user_encoder.transform(time_agg['user_id'])
time_agg['world_idx'] = world_encoder.transform(time_agg['world_id'])

# Merge and create weights
merged = interactions.merge(time_agg[['user_idx', 'world_idx', 'avg_time_minutes']], 
                           on=['user_idx', 'world_idx'], how='left')
merged['avg_time_minutes'] = merged['avg_time_minutes'].fillna(0)
merged['weight'] = merged['action_count'] + 0.1 * merged['avg_time_minutes']

# Create sparse matrix
n_users = len(user_encoder.classes_)
n_worlds = len(world_encoder.classes_)
interaction_matrix = sparse.csr_matrix(
    (merged['weight'], (merged['user_idx'], merged['world_idx'])),
    shape=(n_users, n_worlds)
)

print(f"✅ Interaction Matrix: {n_users} users × {n_worlds} worlds")
print(f"  Non-zero entries: {interaction_matrix.nnz:,}")
print(f"  Density: {interaction_matrix.nnz / (n_users * n_worlds):.2%}")

## Step 6: Interaction Matrix Visualization

In [ ]:
# === Visualize Interaction Matrix Structure ===
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Sample of interaction matrix
ax1 = axes[0]
sample_size = min(100, n_users, n_worlds)
sample_matrix = interaction_matrix[:sample_size, :sample_size].toarray()
im1 = ax1.imshow(sample_matrix, cmap='viridis', aspect='auto', interpolation='nearest')
ax1.set_xlabel('World Index')
ax1.set_ylabel('User Index')
ax1.set_title(f'Interaction Matrix Sample ({sample_size}x{sample_size})')
plt.colorbar(im1, ax=ax1, label='Interaction Weight')

# Sparsity pattern
ax2 = axes[1]
binary_matrix = (sample_matrix > 0).astype(int)
im2 = ax2.imshow(binary_matrix, cmap='Greys', aspect='auto', interpolation='nearest')
ax2.set_xlabel('World Index')
ax2.set_ylabel('User Index')
ax2.set_title('Sparsity Pattern (Black = Interaction)')

# Distribution of interaction weights
ax3 = axes[2]
weights = interaction_matrix.data
ax3.hist(weights, bins=50, edgecolor='black', alpha=0.7, color='darkblue')
ax3.set_xlabel('Interaction Weight')
ax3.set_ylabel('Frequency')
ax3.set_title('Distribution of Interaction Weights')
ax3.set_yscale('log')
ax3.axvline(np.median(weights), color='red', linestyle='--', 
           label=f'Median: {np.median(weights):.2f}')
ax3.legend()

plt.tight_layout()
plt.show()

print("📊 Interaction Matrix Statistics:")
print(f"  Mean weight: {np.mean(weights):.2f}")
print(f"  Median weight: {np.median(weights):.2f}")
print(f"  Max weight: {np.max(weights):.2f}")
print(f"  Weight std dev: {np.std(weights):.2f}")

## Step 7: World Reach Distribution

In [ ]:
# === World Reach Analysis ===
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# World reach distribution
world_reach = (interaction_matrix > 0).sum(axis=0).A1
user_reach = (interaction_matrix > 0).sum(axis=1).A1

ax1 = axes[0]
ax1.hist(world_reach, bins=50, edgecolor='black', alpha=0.7, color='green')
ax1.set_xlabel('Number of Users Reached')
ax1.set_ylabel('Number of Worlds')
ax1.set_title('World Reach Distribution')
ax1.axvline(np.median(world_reach), color='red', linestyle='--',
           label=f'Median: {np.median(world_reach):.0f}')
ax1.legend()

# User diversity distribution
ax2 = axes[1]
ax2.hist(user_reach, bins=50, edgecolor='black', alpha=0.7, color='blue')
ax2.set_xlabel('Number of Worlds Visited')
ax2.set_ylabel('Number of Users')
ax2.set_title('User Diversity Distribution')
ax2.axvline(np.median(user_reach), color='red', linestyle='--',
           label=f'Median: {np.median(user_reach):.0f}')
ax2.legend()

# Reach vs Popularity scatter
ax3 = axes[2]
world_total_weight = interaction_matrix.sum(axis=0).A1
ax3.scatter(world_reach, world_total_weight, alpha=0.5, s=20)
ax3.set_xlabel('Number of Users (Reach)')
ax3.set_ylabel('Total Interaction Weight')
ax3.set_title('World Reach vs Total Engagement')
ax3.set_xscale('log')
ax3.set_yscale('log')

# Add trend line
z = np.polyfit(np.log(world_reach[world_reach > 0]), 
               np.log(world_total_weight[world_reach > 0]), 1)
p = np.poly1d(z)
x_trend = np.logspace(np.log10(world_reach.min()), np.log10(world_reach.max()), 100)
ax3.plot(x_trend, np.exp(p(np.log(x_trend))), 'r-', alpha=0.5, label=f'Trend (slope={z[0]:.2f})')
ax3.legend()

plt.tight_layout()
plt.show()

print("📊 Reach Statistics:")
print(f"  Worlds with 1 user: {(world_reach == 1).sum()}")
print(f"  Worlds with >100 users: {(world_reach > 100).sum()}")
print(f"  Users visiting 1 world: {(user_reach == 1).sum()}")
print(f"  Users visiting >50 worlds: {(user_reach > 50).sum()}")

## Step 8: Feature Engineering & Preparation

In [ ]:
# === PROPER Feature Engineering ===
print("🔧 Feature Engineering Phase")
print("="*50)

# Identify problematic columns to EXCLUDE
EXCLUDE_COLUMNS = [
    'user_id', 'world_id',      # Identifiers
    'action_id',                # Unique per row - LEAKAGE!
    'action_timestamp',         # Raw timestamp - LEAKAGE!
    'user_name', 'world_name',  # Names are quasi-identifiers
    'creator_name'              # Too specific
]

# List features we WILL use
USER_FEATURES = [
    'is_creator', 'region', 'preferred_language', 
    'age_group', 'content_filter_level', 'preferred_categories'
]

WORLD_FEATURES = [
    'category', 'sub_category', 'tags',
    'avg_session_minutes', 'rating', 'total_ratings',
    'likes', 'max_players', 'current_players',
    'maturity_rating', 'is_featured'
]

INTERACTION_FEATURES = [
    'action_type', 'action_value'
]

# Helper function
def encode_categoricals(df, exclude_cols):
    """Safely encode categorical columns"""
    df_encoded = df.copy()
    for col in df_encoded.columns:
        if col not in exclude_cols and df_encoded[col].dtype == 'object':
            le = LabelEncoder()
            df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))
    return df_encoded

# Encode user and world features
users_enc = encode_categoricals(users, ['user_id'])
worlds_enc = encode_categoricals(worlds, ['world_id'])

# Prepare positive samples
print("Creating positive samples...")
positives = actions.copy()
positives = positives.merge(users_enc, on='user_id', how='left')
positives = positives.merge(worlds_enc, on='world_id', how='left')
positives['label'] = 1

# Create negative samples
print("Creating hard negative samples...")
world_popularity_weights = actions.groupby('world_id').size()
world_popularity_weights = world_popularity_weights / world_popularity_weights.sum()

negative_samples = []
n_negatives_needed = len(positives)
all_users = users['user_id'].unique()
all_worlds = worlds['world_id'].unique()
existing_interactions = set(zip(actions['user_id'], actions['world_id']))

np.random.seed(RANDOM_STATE)
attempts = 0
while len(negative_samples) < n_negatives_needed and attempts < n_negatives_needed * 3:
    user = np.random.choice(all_users)
    world = np.random.choice(world_popularity_weights.index, p=world_popularity_weights.values)
    
    if (user, world) not in existing_interactions:
        negative_samples.append({'user_id': user, 'world_id': world})
    attempts += 1

negatives = pd.DataFrame(negative_samples[:n_negatives_needed])
negatives = negatives.merge(users_enc, on='user_id', how='left')
negatives = negatives.merge(worlds_enc, on='world_id', how='left')
negatives['label'] = 0
negatives['action_type'] = 0
negatives['action_value'] = 0

print(f"\n📊 Dataset Balance:")
print(f"  Positive samples: {len(positives):,}")
print(f"  Negative samples: {len(negatives):,}")

## Step 9: Feature Correlation Analysis

In [ ]:
# === Feature Correlation Analysis ===
# Combine positive and negative samples
all_samples = pd.concat([positives, negatives], ignore_index=True)

# Select only our intended features
feature_cols = USER_FEATURES + WORLD_FEATURES + INTERACTION_FEATURES
available_cols = [col for col in feature_cols if col in all_samples.columns]

X_full = all_samples[available_cols]
y_full = all_samples['label']

# Encode any remaining categorical features
X_encoded = X_full.copy()
for col in X_encoded.columns:
    if X_encoded[col].dtype == 'object':
        le = LabelEncoder()
        X_encoded[col] = le.fit_transform(X_encoded[col].astype(str))

# Handle missing values
imputer = SimpleImputer(strategy='mean')
X_encoded_imputed = pd.DataFrame(
    imputer.fit_transform(X_encoded),
    columns=X_encoded.columns
)

# Calculate correlations
correlations = X_encoded_imputed.corrwith(y_full).sort_values(ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Top Feature Correlations
ax1 = axes[0]
top_corr = correlations.head(15)
colors = ['green' if x > 0 else 'red' for x in top_corr.values]
ax1.barh(range(len(top_corr)), top_corr.values, color=colors, alpha=0.7)
ax1.set_yticks(range(len(top_corr)))
ax1.set_yticklabels(top_corr.index)
ax1.set_xlabel('Correlation with Label')
ax1.set_title('Top 15 Feature Correlations with Interaction')
ax1.axvline(0, color='black', linestyle='-', linewidth=0.5)
ax1.grid(True, alpha=0.3, axis='x')

# Feature Correlation Matrix
ax2 = axes[1]
# Select top 10 features for correlation matrix
top_features = correlations.abs().nlargest(10).index
corr_matrix = X_encoded_imputed[top_features].corr()
im = ax2.imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1, aspect='auto')
ax2.set_xticks(range(len(top_features)))
ax2.set_yticks(range(len(top_features)))
ax2.set_xticklabels(top_features, rotation=45, ha='right')
ax2.set_yticklabels(top_features)
ax2.set_title('Feature Correlation Matrix (Top 10)')
plt.colorbar(im, ax=ax2, label='Correlation')

# Correlation Distribution
ax3 = axes[2]
ax3.hist(correlations.values, bins=30, edgecolor='black', alpha=0.7, color='purple')
ax3.set_xlabel('Correlation with Label')
ax3.set_ylabel('Number of Features')
ax3.set_title('Feature Correlation Distribution')
ax3.axvline(0, color='red', linestyle='--', alpha=0.5)
ax3.axvline(-0.8, color='orange', linestyle='--', alpha=0.5, label='Suspicious threshold')
ax3.axvline(0.8, color='orange', linestyle='--', alpha=0.5)
ax3.legend()

plt.tight_layout()
plt.show()

print("🔍 Feature Correlation Analysis:")
print(f"  Strongest positive correlation: {correlations.index[0]} ({correlations.values[0]:.3f})")
print(f"  Strongest negative correlation: {correlations.index[-1]} ({correlations.values[-1]:.3f})")
print(f"  Features with |corr| > 0.5: {(correlations.abs() > 0.5).sum()}")
print(f"  Features with |corr| < 0.1: {(correlations.abs() < 0.1).sum()}")

## Step 10: Train ALS Model with Embeddings Visualization

In [ ]:
# === ALS Model Training ===
import os
os.environ['OPENBLAS_NUM_THREADS'] = '1'

try:
    from implicit.als import AlternatingLeastSquares
    print("✅ ALS library imported successfully!")
    als_available = True
except ImportError:
    print("⚠️ 'implicit' library not installed. Skipping ALS...")
    print("To install: pip install implicit")
    als_available = False

if als_available:
    # Split data for ALS
    def leave_one_out_split(interactions_df, seed=RANDOM_STATE):
        """Leave-one-out split for each user"""
        rng = np.random.RandomState(seed)
        test_rows, train_rows = [], []
        
        for uid, group in interactions_df.groupby('user_idx'):
            indices = group.index.values
            if len(indices) == 1:
                train_rows.append(indices[0])
            else:
                test_idx = rng.choice(indices)
                for idx in indices:
                    if idx == test_idx:
                        test_rows.append(idx)
                    else:
                        train_rows.append(idx)
        
        return interactions_df.loc[train_rows], interactions_df.loc[test_rows]
    
    # Create train/test split
    train_df, test_df = leave_one_out_split(merged)
    
    # Create sparse matrices
    train_matrix = sparse.csr_matrix(
        (train_df['weight'], (train_df['user_idx'], train_df['world_idx'])),
        shape=(n_users, n_worlds)
    )
    test_matrix = sparse.csr_matrix(
        (test_df['weight'], (test_df['user_idx'], test_df['world_idx'])),
        shape=(n_users, n_worlds)
    )
    
    print(f"Training ALS model...")
    print(f"  Train interactions: {train_matrix.nnz:,}")
    print(f"  Test interactions: {test_matrix.nnz:,}")
    
    # Train ALS
    als_model = AlternatingLeastSquares(
        factors=32,
        regularization=0.1,
        iterations=30,
        random_state=RANDOM_STATE,
        use_native=True,
        use_cg=True
    )
    
    # Add confidence weights
    confidence_matrix = (train_matrix * 40).astype('double')
    confidence_matrix.data += 1
    als_model.fit(confidence_matrix, show_progress=True)
    
    print("✅ ALS model trained!")
    
    # Calculate metrics
    def calculate_ranking_metrics(model, train_mat, test_mat, k=10, n_samples=100):
        """Calculate ranking metrics for ALS"""
        precision_scores = []
        recall_scores = []
        ndcg_scores = []
        
        users_to_test = np.random.choice(range(train_mat.shape[0]), 
                                       min(n_samples, train_mat.shape[0]), 
                                       replace=False)
        
        for user_id in users_to_test:
            test_items = set(test_mat[user_id].indices)
            if len(test_items) == 0:
                continue
            
            # Get scores
            user_vec = model.user_factors[user_id]
            scores = user_vec @ model.item_factors.T
            
            # Get top-k recommendations
            top_k_items = np.argsort(-scores)[:k]
            top_k_set = set(top_k_items)
            
            # Calculate metrics
            hits = len(test_items & top_k_set)
            precision = hits / k if k > 0 else 0
            recall = hits / len(test_items) if len(test_items) > 0 else 0
            
            precision_scores.append(precision)
            recall_scores.append(recall)
            
            # NDCG calculation
            relevance = np.array([1 if i in test_items else 0 for i in top_k_items])
            if relevance.sum() > 0:
                dcg = np.sum(relevance / np.log2(np.arange(2, k + 2)))
                idcg = np.sum(np.ones(min(len(test_items), k)) / np.log2(np.arange(2, min(len(test_items), k) + 2)))
                ndcg = dcg / idcg if idcg > 0 else 0
                ndcg_scores.append(ndcg)
        
        return {
            'precision@k': np.mean(precision_scores) if precision_scores else 0,
            'recall@k': np.mean(recall_scores) if recall_scores else 0,
            'ndcg@k': np.mean(ndcg_scores) if ndcg_scores else 0
        }
    
    als_metrics = calculate_ranking_metrics(als_model, train_matrix, test_matrix, k=10)
    
    # Simple AUC calculation
    def evaluate_als_auc(model, train_mat, test_mat, n_samples=100):
        scores = []
        users_to_test = np.random.choice(range(train_mat.shape[0]), 
                                       min(n_samples, train_mat.shape[0]), 
                                       replace=False)
        
        for user_id in users_to_test:
            test_items = test_mat[user_id].indices
            if len(test_items) == 0:
                continue
            
            user_vec = model.user_factors[user_id]
            scores_all = user_vec @ model.item_factors.T
            
            labels = np.zeros(len(scores_all))
            labels[test_items] = 1
            
            if len(np.unique(labels)) > 1:
                auc = roc_auc_score(labels, scores_all)
                scores.append(auc)
        
        return np.mean(scores) if scores else 0.5
    
    als_auc = evaluate_als_auc(als_model, train_matrix, test_matrix)
    
    print(f"\n📊 ALS Performance:")
    print(f"  ROC-AUC: {als_auc:.3f}")
    print(f"  Precision@10: {als_metrics['precision@k']:.3f}")
    print(f"  Recall@10: {als_metrics['recall@k']:.3f}")
    print(f"  NDCG@10: {als_metrics['ndcg@k']:.3f}")
else:
    als_auc = 0.5
    als_metrics = {'precision@k': 0, 'recall@k': 0, 'ndcg@k': 0}

## Step 11: User Embeddings Visualization (t-SNE)

In [ ]:
# === Visualize User Embeddings with t-SNE ===
if als_available and 'als_model' in locals():
    print("Generating t-SNE visualization of user embeddings...")
    
    # Sample users for visualization
    n_sample_users = min(500, n_users)
    sample_indices = np.random.choice(n_users, n_sample_users, replace=False)
    
    # Get user embeddings
    user_embeddings = als_model.user_factors[sample_indices]
    
    # Calculate user activity levels for coloring
    user_activities = np.array([train_matrix[i].nnz for i in sample_indices])
    
    # Apply t-SNE
    tsne = TSNE(n_components=2, random_state=RANDOM_STATE, perplexity=30)
    embeddings_2d = tsne.fit_transform(user_embeddings)
    
    # Create visualization
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # t-SNE plot colored by activity
    ax1 = axes[0]
    scatter = ax1.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], 
                         c=user_activities, cmap='viridis', 
                         s=20, alpha=0.6)
    ax1.set_xlabel('t-SNE Component 1')
    ax1.set_ylabel('t-SNE Component 2')
    ax1.set_title('User Embeddings (t-SNE Projection)')
    plt.colorbar(scatter, ax=ax1, label='User Activity Level')
    
    # Density plot
    ax2 = axes[1]
    if sns:
        x, y = embeddings_2d[:, 0], embeddings_2d[:, 1]
        from scipy.stats import gaussian_kde
        xy = np.vstack([x, y])
        z = gaussian_kde(xy)(xy)
        idx = z.argsort()
        x, y, z = x[idx], y[idx], z[idx]
        scatter2 = ax2.scatter(x, y, c=z, s=20, cmap='plasma', alpha=0.6)
        ax2.set_xlabel('t-SNE Component 1')
        ax2.set_ylabel('t-SNE Component 2')
        ax2.set_title('User Embedding Density')
        plt.colorbar(scatter2, ax=ax2, label='Density')
    else:
        ax2.hist2d(embeddings_2d[:, 0], embeddings_2d[:, 1], bins=30, cmap='plasma')
        ax2.set_xlabel('t-SNE Component 1')
        ax2.set_ylabel('t-SNE Component 2')
        ax2.set_title('User Embedding Density')
        plt.colorbar(ax2.collections[0], ax=ax2, label='Count')
    
    plt.tight_layout()
    plt.show()
    
    print("✅ User embeddings visualization complete!")
else:
    print("⚠️ ALS model not available for embedding visualization")

## Step 12: Train Random Forest Model

In [ ]:
# === Random Forest Training ===
print("🌲 Training Random Forest...")
print("="*50)

# Split data using the full dataset
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded_imputed, y_full, test_size=0.2, random_state=RANDOM_STATE, stratify=y_full
)

print(f"Training set: {len(X_train):,} samples")
print(f"Test set: {len(X_test):,} samples")
print(f"Features: {X_train.shape[1]}")

# Train Random Forest
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf.fit(X_train, y_train)

# Get predictions
train_pred = rf.predict_proba(X_train)[:, 1]
test_pred = rf.predict_proba(X_test)[:, 1]

# Calculate comprehensive metrics
def calculate_rf_metrics(y_true, y_pred, threshold=0.5):
    y_pred_binary = (y_pred > threshold).astype(int)
    
    # Sort predictions for ranking metrics
    sorted_indices = np.argsort(-y_pred)
    top_10_pct = int(len(y_pred) * 0.1)
    top_10_indices = sorted_indices[:top_10_pct]
    
    metrics = {
        'roc_auc': roc_auc_score(y_true, y_pred),
        'avg_precision': average_precision_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred_binary),
        'recall': recall_score(y_true, y_pred_binary),
        'precision@10': y_true.iloc[top_10_indices].mean() if hasattr(y_true, 'iloc') else y_true[top_10_indices].mean(),
        'recall@10': y_true.iloc[top_10_indices].sum() / y_true.sum() if hasattr(y_true, 'iloc') else y_true[top_10_indices].sum() / y_true.sum()
    }
    
    # NDCG@10
    relevance_scores = y_true.iloc[sorted_indices[:10]].values if hasattr(y_true, 'iloc') else y_true[sorted_indices[:10]]
    if relevance_scores.sum() > 0:
        dcg = np.sum(relevance_scores / np.log2(np.arange(2, 12)))
        idcg = np.sum(np.ones(min(y_true.sum(), 10)) / np.log2(np.arange(2, min(y_true.sum(), 10) + 2)))
        metrics['ndcg@10'] = dcg / idcg if idcg > 0 else 0
    else:
        metrics['ndcg@10'] = 0
    
    return metrics

rf_train_metrics = calculate_rf_metrics(y_train, train_pred)
rf_test_metrics = calculate_rf_metrics(y_test, test_pred)

print(f"\n📊 Random Forest Performance:")
print("Training Metrics:")
for metric, value in rf_train_metrics.items():
    print(f"  {metric}: {value:.3f}")
print("\nTest Metrics:")
for metric, value in rf_test_metrics.items():
    print(f"  {metric}: {value:.3f}")

# Check for overfitting
overfitting_gap = rf_train_metrics['roc_auc'] - rf_test_metrics['roc_auc']
if overfitting_gap > 0.1:
    print(f"\n⚠️ Warning: Possible overfitting (gap: {overfitting_gap:.3f})")
else:
    print(f"\n✅ Good generalization (gap: {overfitting_gap:.3f})")

## Step 13: Comprehensive Model Comparison

In [ ]:
# === Comprehensive Model Comparison ===
print("🏆 COMPREHENSIVE MODEL COMPARISON")
print("="*60)

# Prepare comparison data
comparison_data = {
    'Metric': ['ROC-AUC', 'Avg Precision', 'Precision@10', 'Recall@10', 'NDCG@10'],
    'ALS (Collaborative)': [
        als_auc,
        0.0,  # ALS doesn't directly provide avg precision
        als_metrics['precision@k'],
        als_metrics['recall@k'],
        als_metrics['ndcg@k']
    ],
    'Random Forest (Content)': [
        rf_test_metrics['roc_auc'],
        rf_test_metrics['avg_precision'],
        rf_test_metrics['precision@10'],
        rf_test_metrics['recall@10'],
        rf_test_metrics['ndcg@10']
    ]
}

comparison_df = pd.DataFrame(comparison_data)

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Bar chart comparison
ax1 = axes[0]
x = np.arange(len(comparison_df))
width = 0.35
ax1.bar(x - width/2, comparison_df['ALS (Collaborative)'], width, 
       label='ALS', color='steelblue', alpha=0.8)
ax1.bar(x + width/2, comparison_df['Random Forest (Content)'], width, 
       label='Random Forest', color='darkorange', alpha=0.8)
ax1.set_xlabel('Metric')
ax1.set_ylabel('Score')
ax1.set_title('Model Performance Comparison')
ax1.set_xticks(x)
ax1.set_xticklabels(comparison_df['Metric'], rotation=45, ha='right')
ax1.legend()
ax1.grid(True, alpha=0.3, axis='y')
ax1.set_ylim(0, 1)

# Radar chart
ax2 = axes[1]
angles = np.linspace(0, 2 * np.pi, len(comparison_df), endpoint=False)
angles = np.concatenate((angles, [angles[0]]))

als_values = comparison_df['ALS (Collaborative)'].values
als_values = np.concatenate((als_values, [als_values[0]]))
rf_values = comparison_df['Random Forest (Content)'].values
rf_values = np.concatenate((rf_values, [rf_values[0]]))

ax2 = plt.subplot(1, 3, 2, projection='polar')
ax2.plot(angles, als_values, 'o-', linewidth=2, label='ALS', color='steelblue')
ax2.fill(angles, als_values, alpha=0.25, color='steelblue')
ax2.plot(angles, rf_values, 'o-', linewidth=2, label='Random Forest', color='darkorange')
ax2.fill(angles, rf_values, alpha=0.25, color='darkorange')
ax2.set_xticks(angles[:-1])
ax2.set_xticklabels(comparison_df['Metric'], size=10)
ax2.set_ylim(0, 1)
ax2.set_title('Model Performance Radar Chart', pad=20)
ax2.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
ax2.grid(True)

# Metric importance/weight visualization
ax3 = axes[2]
metric_weights = {'ROC-AUC': 0.25, 'Avg Precision': 0.2, 
                 'Precision@10': 0.2, 'Recall@10': 0.15, 'NDCG@10': 0.2}
weighted_als = sum(comparison_df['ALS (Collaborative)'][i] * list(metric_weights.values())[i] 
                  for i in range(len(metric_weights)))
weighted_rf = sum(comparison_df['Random Forest (Content)'][i] * list(metric_weights.values())[i] 
                 for i in range(len(metric_weights)))

bars = ax3.bar(['ALS\n(Collaborative)', 'Random Forest\n(Content)'], 
              [weighted_als, weighted_rf], 
              color=['steelblue', 'darkorange'], alpha=0.8)
ax3.set_ylabel('Weighted Score')
ax3.set_title('Overall Weighted Performance')
ax3.set_ylim(0, 1)

# Add value labels on bars
for bar, score in zip(bars, [weighted_als, weighted_rf]):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{score:.3f}', ha='center', fontweight='bold')

# Add weight information
weight_text = '\n'.join([f"{k}: {v:.0%}" for k, v in metric_weights.items()])
ax3.text(1.5, 0.5, f'Weights:\n{weight_text}', 
        transform=ax3.transData, fontsize=9,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

# Print detailed comparison table
print("\n📊 Detailed Metrics Comparison:")
print(comparison_df.to_string(index=False))

print("\n📝 Model Characteristics:")
print("="*60)
print("ALS (Collaborative Filtering):")
print("  ✓ Learns latent patterns from user-item interactions")
print("  ✓ No feature engineering required")
print("  ✗ Suffers from cold start problem")
print(f"  ✗ Struggles with high sparsity ({sparsity:.1%})")

print("\nRandom Forest (Content-Based):")
print("  ✓ Handles cold start well")
print("  ✓ Interpretable feature importance")
print("  ✓ Works with sparse data")
print("  ✗ Requires careful feature engineering")
print("  ✗ May miss complex interaction patterns")

print("\n🎯 Recommendation:")
if weighted_rf > weighted_als:
    print(f"  Random Forest performs better overall ({weighted_rf:.3f} vs {weighted_als:.3f})")
else:
    print(f"  ALS performs better overall ({weighted_als:.3f} vs {weighted_rf:.3f})")
print("  Consider hybrid approach for production deployment")

## Step 14: Feature Importance Deep Dive

In [ ]:
# === Feature Importance Analysis ===
feature_importance = pd.DataFrame({
    'feature': X_encoded_imputed.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Top features bar plot
ax1 = axes[0, 0]
top_features = feature_importance.head(15)
ax1.barh(range(len(top_features)), top_features['importance'], color='teal', alpha=0.7)
ax1.set_yticks(range(len(top_features)))
ax1.set_yticklabels(top_features['feature'])
ax1.set_xlabel('Importance')
ax1.set_title('Top 15 Feature Importances')
ax1.grid(axis='x', alpha=0.3)

# Cumulative importance
ax2 = axes[0, 1]
cumsum = feature_importance['importance'].cumsum() / feature_importance['importance'].sum()
ax2.plot(range(1, len(cumsum) + 1), cumsum, marker='o', markersize=3, color='darkblue')
ax2.axhline(0.8, color='red', linestyle='--', alpha=0.5, label='80% variance')
ax2.axhline(0.9, color='orange', linestyle='--', alpha=0.5, label='90% variance')
ax2.set_xlabel('Number of Features')
ax2.set_ylabel('Cumulative Importance')
ax2.set_title('Cumulative Feature Importance')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Feature importance by category
ax3 = axes[1, 0]
feature_categories = {
    'User': [f for f in feature_importance['feature'] if f in USER_FEATURES],
    'World': [f for f in feature_importance['feature'] if f in WORLD_FEATURES],
    'Interaction': [f for f in feature_importance['feature'] if f in INTERACTION_FEATURES]
}
category_importance = {}
for cat, features in feature_categories.items():
    category_importance[cat] = feature_importance[feature_importance['feature'].isin(features)]['importance'].sum()

ax3.pie(category_importance.values(), labels=category_importance.keys(), 
       autopct='%1.1f%%', colors=['skyblue', 'lightcoral', 'lightgreen'],
       startangle=90)
ax3.set_title('Feature Importance by Category')

# Feature importance distribution
ax4 = axes[1, 1]
ax4.hist(feature_importance['importance'], bins=30, edgecolor='black', 
        alpha=0.7, color='purple')
ax4.set_xlabel('Feature Importance')
ax4.set_ylabel('Number of Features')
ax4.set_title('Feature Importance Distribution')
ax4.axvline(feature_importance['importance'].median(), color='red', 
           linestyle='--', label=f"Median: {feature_importance['importance'].median():.4f}")
ax4.legend()

plt.tight_layout()
plt.show()

# Find how many features explain 80% of importance
n_features_80 = (cumsum < 0.8).sum() + 1
n_features_90 = (cumsum < 0.9).sum() + 1

print("📊 Feature Importance Analysis:")
print(f"  Total features: {len(feature_importance)}")
print(f"  Features for 80% importance: {n_features_80}")
print(f"  Features for 90% importance: {n_features_90}")
print(f"  Features with <1% importance: {(feature_importance['importance'] < 0.01).sum()}")
print(f"\n  Category Importance:")
for cat, imp in category_importance.items():
    print(f"    {cat}: {imp:.3f} ({imp/feature_importance['importance'].sum():.1%})")

## Step 15: Final Summary and Recommendations

In [ ]:
# === Final Summary ===
print("🎯 FINAL SUMMARY & RECOMMENDATIONS")
print("="*60)

print("\n1. DATA CHARACTERISTICS:")
print(f"   - Dataset sparsity: {sparsity:.1%}")
print(f"   - User concentration (Gini): {gini_user:.3f}")
print(f"   - World concentration (Gini): {gini_world:.3f}")
print(f"   - Average session duration: {session_durations.mean():.1f} minutes")

print("\n2. MODEL PERFORMANCE:")
print("   ALS (Collaborative):")
print(f"     - ROC-AUC: {als_auc:.3f}")
print(f"     - Precision@10: {als_metrics['precision@k']:.3f}")
print(f"     - NDCG@10: {als_metrics['ndcg@k']:.3f}")
print("   Random Forest (Content):")
print(f"     - ROC-AUC: {rf_test_metrics['roc_auc']:.3f}")
print(f"     - Precision@10: {rf_test_metrics['precision@10']:.3f}")
print(f"     - NDCG@10: {rf_test_metrics['ndcg@10']:.3f}")

print("\n3. KEY INSIGHTS:")
print(f"   - Top 10% users generate {user_sorted.head(int(len(user_sorted)*0.1))['num_actions'].sum() / user_sorted['num_actions'].sum():.1%} of actions")
print(f"   - Top 10% worlds receive {world_sorted.head(int(len(world_sorted)*0.1))['num_visitors'].sum() / world_sorted['num_visitors'].sum():.1%} of visits")
print(f"   - Most important feature: {feature_importance.iloc[0]['feature']}")
print(f"   - {n_features_80} features explain 80% of model importance")

print("\n4. PRODUCTION RECOMMENDATIONS:")
print("   A. Hybrid Approach:")
print("      - Use RF for cold users (< 5 interactions)")
print("      - Use ALS for warm users (>= 5 interactions)")
print("      - Blend predictions with weighted average")

print("\n   B. Data Collection:")
print("      - Collect more implicit feedback signals")
print("      - Track dwell time and scroll depth")
print("      - Implement session replay for behavior analysis")

print("\n   C. Model Improvements:")
print("      - Consider deep learning (Neural CF, Wide & Deep)")
print("      - Implement online learning for real-time updates")
print("      - Add contextual bandits for exploration/exploitation")

print("\n   D. Evaluation:")
print("      - Implement A/B testing framework")
print("      - Track business metrics (CTR, conversion, retention)")
print("      - Monitor model drift and retrain periodically")

print("\n✅ Analysis complete! All visualizations and metrics generated successfully.")